In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 1. Load the dataset (using latin1 encoding for special characters)
df = pd.read_csv('/content/DAV FINAL PRODUCT BASED PROJECT DATASET (2).csv', encoding='latin1')

# 2. Parse dates and extract the year
df['Order_Date'] = pd.to_datetime(df['Order_Date'], format='%d/%m/%Y', errors='coerce')
df['Year'] = df['Order_Date'].dt.year

# 3. Aggregate data to create a multivariate dataset
# We group by Customer and Year to generate multiple numeric metrics for our charts
df_agg = df.groupby(['Customer_ID', 'Customer_Name', 'Segment', 'Year']).agg(
    Total_Sales=('Sales', 'sum'),
    Items_Purchased=('Product_ID', 'count'),
    Avg_Sale=('Sales', 'mean')
).reset_index()
df_agg.head()

,Customer_ID,Customer_Name,Segment,Year,Total_Sales,Items_Purchased,Avg_Sale
0,AA-10315,Alex Avila,Consumer,2015,756.048,4,189.012
1,AA-10315,Alex Avila,Consumer,2016,26.960,1,26.960
2,AA-10315,Alex Avila,Consumer,2017,4406.072,4,1101.518
3,AA-10315,Alex Avila,Consumer,2018,374.480,2,187.240
4,AA-10375,Allen Armold,Consumer,2015,50.792,2,25.396


1. Univariate Histogram with Range Slider

In [3]:
fig1 = px.histogram(
    df_agg,
    x='Total_Sales',
    title='Task 1.1: Histogram of Total Customer Sales (per year)'
)
fig1.update_layout(xaxis_rangeslider_visible=True)
fig1.show()

2. Multivariate Scatter Plot (Log Scale)

In [4]:
fig2 = px.scatter(
    df_agg,
    x='Items_Purchased',
    y='Total_Sales',
    color='Segment',
    hover_data=['Customer_Name', 'Year'],
    log_y=True,  # Handles the right-skew of high-spending customers
    title='Task 1.2: Items Purchased vs. Total Sales (Log Scale) by Segment'
)
fig2.show()

3. Interactive Box Plot

In [5]:
fig3 = px.box(
    df_agg,
    x='Segment',
    y='Total_Sales',
    points='all',
    title='Task 1.3: Box Plot of Total Sales by Customer Segment'
)
fig3.show()

4. 3D Scatter Plot

In [10]:
fig4 = px.scatter_3d(
    df_agg,
    x='Items_Purchased',
    y='Avg_Sale',
    z='Total_Sales',
    color='Segment',
    title='Task 1.4: 3D Scatter Plot (Items, Average Sale, Total Sales)'
)
# Make points slightly smaller for better 3D visibility
fig4.update_traces(marker=dict(size=4))
fig4.show()

Task 2: Animation and Dropdown Controls
1. Animated Scatter Plot (Frame by Year)

In [7]:
# Sort by year to ensure frames play in chronological order (2015 -> 2018)
df_sorted = df_agg.sort_values('Year')

fig5 = px.scatter(
    df_sorted,
    x='Items_Purchased',
    y='Total_Sales',
    animation_frame='Year',
    size='Avg_Sale',      # Bubble size based on average sale price
    color='Segment',
    hover_name='Customer_Name',
    # Fix axis ranges so the frame doesn't jump during animation
    range_x=[0, df_agg['Items_Purchased'].max() + 2],
    range_y=[0, df_agg['Total_Sales'].max() + 500],
    size_max=40,
    title='Task 2.1: Animated Customer Sales vs Items Purchased (2015-2018)'
)
fig5.show()

2. Dropdown Menu (Switch Y-Axis Metric)

In [8]:
fig6 = go.Figure()

# Add traces. Only the first trace (Total Sales) is visible by default
fig6.add_trace(go.Scatter(x=df_agg['Items_Purchased'], y=df_agg['Total_Sales'],
                          mode='markers', name='Total Sales'))
fig6.add_trace(go.Scatter(x=df_agg['Items_Purchased'], y=df_agg['Avg_Sale'],
                          mode='markers', name='Average Sale', visible=False))

fig6.update_layout(
    title="Task 2.2: Items Purchased vs Selected Sales Metric",
    xaxis_title="Items Purchased",
    updatemenus=[
        dict(
            active=0,
            buttons=list([
                dict(label="Total Sales",
                     method="update",
                     args=[{"visible": [True, False]}, {"yaxis": {"title": "Total Sales ($)"}}]),
                dict(label="Average Sale",
                     method="update",
                     args=[{"visible": [False, True]}, {"yaxis": {"title": "Average Sale ($)"}}])
            ]),
        )
    ]
)
fig6.show()